In [1]:
import duckdb
import pandas as pd

In [2]:
con = duckdb.connect()

con.sql(""" 
    CREATE OR REPLACE VIEW reports AS SELECT * FROM '../data/processed/reports.parquet';
    CREATE OR REPLACE VIEW packages AS SELECT * FROM '../data/processed/packages.parquet';    
        """)

con.sql("SELECT COUNT(*) FROM reports")

count_star()BIGINT
532


In [3]:
atc_counts = con.sql("""
SELECT atc_group, COUNT(atc_group) as n_reports
FROM reports
WHERE atc_group IS NOT NULL
GROUP BY atc_group
ORDER BY n_reports DESC


""").df()

display(atc_counts)

,atc_group,n_reports
0,Nervous system,159
1,Cardiovascular system,74
2,Antineoplastic and immunomodulating agents,64
3,Anti-infectives,50
4,Alimentary tract and metabolism,35
5,Blood,26
6,Systemic hormones,24
7,Sensory organs,21
8,Genito-urinary system and sex hormones,18
9,Various,14


In [4]:
df = pd.read_parquet('../data/processed/reports.parquet')

pandas_counts = df['atc_group'].value_counts().sort_index()

sql_counts = atc_counts.set_index('atc_group')['n_reports'].sort_index()

pd.testing.assert_series_equal(sql_counts, pandas_counts, check_names=False)

In [5]:
critical_share = con.sql("""
SELECT atc_group,COUNT(*) AS n_reports, AVG(CASE WHEN classification = 'supply_critical' THEN 1 ELSE 0 END) as critical_share
FROM reports
WHERE atc_group IS NOT NULL AND classification IS NOT NULL
GROUP BY atc_group
ORDER BY critical_share DESC


""").df()

display(critical_share)

,atc_group,n_reports,critical_share
0,Antiparasitic products,1,1.000000
1,Anti-infectives,50,0.640000
2,Antineoplastic and immunomodulating agents,64,0.625000
3,Various,14,0.428571
4,Systemic hormones,24,0.416667
5,Genito-urinary system and sex hormones,18,0.388889
6,Musculo-skeletal system,11,0.181818
7,Sensory organs,21,0.142857
8,Alimentary tract and metabolism,34,0.117647
9,Nervous system,159,0.113208


In [6]:
pandas_critical = pd.crosstab(
    index=df['atc_group'],
    columns=df['classification'],
    normalize='index'
)['supply_critical'].sort_index()


sql_critical = (
    critical_share
    .set_index('atc_group')['critical_share']
    .sort_index()
)
pd.testing.assert_series_equal(
    pandas_critical,
    sql_critical,
    check_names=False
)

In [7]:
package_rows = con.sql("""
SELECT r.report_id, r.n_packages, COUNT(*) as counted_packages
FROM reports AS r
JOIN packages as p ON p.report_id = r.report_id
GROUP BY r.report_id, r.n_packages
HAVING COUNT(*) <> r.n_packages

""").df()

assert len(package_rows) == 0 

In [8]:
top_companies = con.sql("""
WITH company_shares AS(
    SELECT atc_group, licensee, COUNT(*) as n_reports
    FROM reports
    WHERE atc_group IS NOT NULL
    GROUP BY licensee, atc_group
)
SELECT *,
RANK() OVER  (
    PARTITION BY atc_group
    ORDER BY n_reports DESC
) as rank_in_group 
FROM company_shares
QUALIFY rank_in_group = 1
ORDER BY n_reports DESC
""").df()
pd.set_option('display.max_colwidth', None)
display(top_companies)

,atc_group,licensee,n_reports,rank_in_group
0,Nervous system,1 A Pharma GmbH,16,1
1,Cardiovascular system,Basics GmbH,16,1
2,Antineoplastic and immunomodulating agents,STADAPHARM GmbH,12,1
3,Sensory organs,URSAPHARM Arzneimittel GmbH,9,1
4,Alimentary tract and metabolism,Basics GmbH,6,1
5,Anti-infectives,Dr. Friedrich Eberth Arzneimittel GmbH,5,1
6,Blood,Fresenius Kabi Deutschland GmbH,5,1
7,Systemic hormones,Hexal AG,4,1
8,Respiratory system,PB Pharma GmbH,4,1
9,Dermatologicals,ratiopharm GmbH,3,1


## Summary

All queries run with DuckDB directly on the cleaned parquet files (`reports` and `packages` as views). Every result that also exists in notebook 02 was checked against pandas with `pd.testing.assert_series_equal`.

| Query | SQL features | Check |
|---|---|---|
| Reports per drug class | `GROUP BY`, `COUNT`, `ORDER BY` | matches pandas `value_counts()` |
| Supply-critical share per class | `AVG(CASE WHEN ...)` | matches the pandas crosstab |
| Packages per report | `JOIN`, `GROUP BY`, `HAVING` | 0 mismatches with `n_packages` |
| Top company per drug class | CTE (`WITH`), `RANK() OVER (PARTITION BY ...)`, `QUALIFY` | ties kept (3 companies share rank 1 in genito-urinary) |

**What the cross-checks caught:**
- **Missing values:** `CASE WHEN` counts a missing classification as "not critical", while the pandas crosstab skips it. One report without a classification changed the alimentary share (0.11 vs 0.12) until it was excluded in both.
- **Rounding:** DuckDB's `ROUND` rounds 0.625 up to 0.63; pandas rounds half to even (0.62). Comparisons are therefore done on unrounded values.

**Top companies by class:** 1 A Pharma leads in nervous system drugs (16 reports), Basics GmbH in cardiovascular drugs (16), STADAPHARM in cancer drugs (12).

Note: `QUALIFY` exists in DuckDB, Snowflake and BigQuery, but not in Postgres, where the ranking needs a subquery.